# Social Computing: Notebook 4

**Deadline:** Wednesday, 15th April 2026, 12:00

### Preliminary information:
If you copy another group’s code in Jupyter Notebooks or give someone
your code to copy, both pairs involved in this behavior will get 0
points for the Notebook. This doesn’t mean you can’t consult someone
from another pair in case you and your coding partner are stuck on a
particular part of code though! You can do it, but you need to disclose
this in the Notebook. You can write something like this in a comment
next to the part of code on which you consulted someone - “we both
couldn’t find a solution for this on our own, so we asked students X and
Y for help. They explained us how this works. Now we see the issue was
that we were doing A instead of B”. Describe what prevented you from
figuring out stuff on your own to demonstrate you understood what’s
going on, not just copied code from another pair. If you provide a
proper explanation, you won’t lose the points for this.

#### Deliverables:

1. This notebook with your code and explanations/answers

- Please make sure to remove any code that you’ve used for testing and are not part of your solution, but keep the output cells.
- Rename this notebook file as described below.

**&rArr; You both have to submit your solution on OLAT, but you should submit the same files(s).**

#### How will you be graded?

You will be graded out of 100 points.

---

Please include your names below and edit the name of the file to include your names (e.g. SC_Notebook4_Hamilton_Turing.ipynb):
- Alina Vanessa Brüllhardt
- Elias Mithanios Iskander


---

# Agent-based Modelling

## 1) Short intro on plane boarding and deboarding (10 points)

Fast boarding and deboarding times are important for both airlines and passengers. For airlines, quick boarding is key because short delays could result in a departure rescheduling for arbitrary later times. In addition, crew members and planes are often scheduled for multiple flights per day; therefore, a delay in one fight might have consequences in other flights too. Similarly, passengers benefit from efficient (de)boarding methods as well. Besides a lack of excitement for long queueing times, delays could result in loss of connections. Therefore, efficient (de)boarding methods are desirable by both airlines and passengers.

In this notebook, we will create an agent-based model to compare different boarding and deboarding methods. More precisely, we aim to find out which would be the most efficient (de)boarding methods.

We start by some short question on the background:

**Exercise 1.1 (5 points)**
We mentioned in the first paragraph why efficiency in (de)boarding is important for both airlines and passengers. Are there any other parties that might benefit from such efficiency? If yes which and in what ways?
You can use the space below to answer this question.

**Exercise 1.2 (5 points)**
The efficiency in (de)boarding is not the most important factor for passengers and companies. Use the space below to give 2 examples of competing interests. Don’t forget to explain why these interests are at stake with the (de)boarding time efficiency.

## 2) Plane boarding (55 points)
The next 4 code cells have key code for running a plane-boarding simulation. They contain the following:

1.	**Parameter values:** In this notebook, we have used the following plane [configuration](https://seatmaps.com/airlines/ba-british-airways/airbus-a320/). Moreover, there is one extra space on the aisle at the entrance, and two extra space at the end of the aisle. These spaces correspond to existing spaces in an aircraft and are convenient to consider to have a cleaner code. This code cell contains global variables and enumerations.
2.	**Passengers:** This class models the important aspects of a passenger. Passengers are labeled by their seat number.
3.	**Plane:** This class contains the configuration and occupancy of a plane (how does a plane look at one moment of time).
4.	**Simulation:** This class runs a simulation; i.e. it creates a plane, orders the passengers, and iterates the plane configuration until everybody is seated.

Take a moment to familiarize yourself with the code. Don’t be intimidated by its length; you can use the comments to help you better understand what the simulation does. 


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from copy import deepcopy
from typing import Any
from enum import StrEnum

# -------- Fixed parameters --------
NUM_SEATS_PER_ROW_PER_SIDE = 3
NUM_SIDES = 2

# -------- Variable parameters --------
# the number of rows in the plane
# note that in the configuration linked above, there are 28 rows; however, for initial runs we suggest to use a smaller number
NUM_ROWS = 5
# the number of iterations it takes an agent to load their luggage
LUGGAGE_LOADING_TIME = 1

# -------- Parameters that depend on other values --------
MUM_SEATS = NUM_ROWS * NUM_SEATS_PER_ROW_PER_SIDE * NUM_SIDES

# -------- Enumerations --------
class Goal(StrEnum):
    FIND_SEAT = 'find_seat'
    GO_TO_SEAT = 'go_to_seat'
    MAKE_SPACE = 'make_space'
    SEATED = 'seated'

class Itinerary(StrEnum):
    LOAD_LUGGAGE = 'load_luggage'
    UNLOAD_LUGGAGE = 'unload_luggage'

In [ ]:
class Passenger:
    def __init__ (self, seat_no: int) -> None:
        # id = unique number between 1 and the number of seats
        # this corresponds to their seat number (-1 --> unassigned)
        self.id: int = seat_no
        
        # row number (starting from 1)
        self.row_no: int = int(seat_no / (NUM_SEATS_PER_ROW_PER_SIDE * NUM_SIDES)) + 1
        
        # the seat number within the row
        self.column_no: int = seat_no % (NUM_SEATS_PER_ROW_PER_SIDE * NUM_SIDES)
        # if the seat is across the aisle, add 1 to its number
        if self.column_no >= NUM_SEATS_PER_ROW_PER_SIDE:
            self.column_no += 1
                       
        # luggage loading status (initially, this is how long it takes an agent to load their luggage in the overhead compartment)
        self.remaining_loading_luggage: int = LUGGAGE_LOADING_TIME
        
        # itinerary - what they have to do to get to their seat
        self.itinerary: list[tuple[int, int] | Itinerary] = []

        # 1) advance on the aisle, i.e. move up, i.e. add (0, 1) to their position
        self.itinerary = [(1, 0) for _i in range(self.row_no)]

        # 2) load the luggage
        self.itinerary.append(Itinerary.LOAD_LUGGAGE)

        # 3) get to seat (advance horizontally)
        if self.column_no < NUM_SEATS_PER_ROW_PER_SIDE:
            self.itinerary.extend([(0, -1) for _i in range(NUM_SEATS_PER_ROW_PER_SIDE - self.column_no)])
        else:
            self.itinerary.extend([(0, 1) for _i in range(self.column_no - NUM_SEATS_PER_ROW_PER_SIDE)])
        
        # which squares need to be empty for them to access their row number, start loading, and get to seat
        # labeled from aisle to seat number
        self.needed_free: list[tuple[int, int]] = []
        if self.column_no < NUM_SEATS_PER_ROW_PER_SIDE:
            self.needed_free = [(self.row_no, i) for i in range(NUM_SEATS_PER_ROW_PER_SIDE, self.column_no, -1)]
        else:
            self.needed_free = [(self.row_no, i) for i in range(NUM_SEATS_PER_ROW_PER_SIDE, self.column_no)]
        
        # goal: find_seat/go_to_seat/make_space/seated
        # find_seat = before needing to load the luggage - need to find their row
        # go_to_seat = know their row (already loaded luggage) - need to proceed to seat
        # make_space = need to exit their seat to make space for somebody else to enter
        # seated = empty itinerary, none of the above
        self.goal: Goal = Goal.FIND_SEAT

    def __str__(self):
        return str(self.id)
    
    def __eq__ (self, other):
        """Two passengers are equal if they have the same seat"""
        if isinstance(other, Passenger):
            return self.id == other.id
        else:
            return False
    
    def __hash__(self):
        """Passengers are hashed by their id"""
        return self.id

In [ ]:
class Plane:
    def __init__(self):
        # passengers are labeled by their seat number
        self.passengers: list[Passenger] = [Passenger(i) for i in range(MUM_SEATS)]

        # frequency vector of passengers that moved this round (in one iteration a passenger can make at most one move)
        self.moved: dict[Passenger, int] = {self.passengers[i]:0 for i in range(MUM_SEATS)}
        
        # boarding order
        self.boarding_order: list[Passenger] = []
        
        # current plane occupation
        # None --> that space is empty
        # there is a row 0 (one extra space in front) and two rows at the end (for aisle space - no seats)
        self.num_rows_map: int = NUM_ROWS + 3
        self.num_cols_map: int = NUM_SEATS_PER_ROW_PER_SIDE * NUM_SIDES + 1
        self.map: list[list[Passenger | None]] = [[None for _c in range(self.num_cols_map)] for _r in range(self.num_rows_map)]
    
    def print_map(self) -> None:
        """Prints the current plane map."""
        for r in range(self.num_rows_map):
            current_row = ""
            for c in range(self.num_cols_map):
                if self.map[r][c]:
                    l = len(str(self.map[r][c]))
                    spaces = "  "
                    for i in range(3 - l):
                        spaces += " "
                    current_row += spaces + str(self.map[r][c])
                elif r in [0, self.num_rows_map - 2, self.num_rows_map - 1] and c != NUM_SEATS_PER_ROW_PER_SIDE:
                    # these are unaccessible places
                    current_row += "  xxx"
                else:
                    current_row += "  ___"
            print(current_row)
    
    def reset_moved(self) -> None:
        """Resets all moved to zero"""
        
        for p in self.moved:
            self.moved[p] = 0
            
    def new_passenger_in_plane(self) -> bool:
        """If there is an empty seat at the entry of the aisle, a new agent enters the plane"""
        
        if len(self.boarding_order) == 0 or self.map[0][NUM_SEATS_PER_ROW_PER_SIDE]:
            return False
        else:
            # next passenger in line enters the plane
            self.map[0][NUM_SEATS_PER_ROW_PER_SIDE] = self.boarding_order[0]
            
            # that passenger is removed from the boarding queue
            self.boarding_order.pop(0)
            
            return True
    
    def move(self, passenger_position: tuple[int, int]) -> bool:
        """The passenger currently located on the position passenger_position  either loads their luggage, or, they advance on the corridor (depending on the situation they are in).
        They do one step in their itinerary."""
                              
        row, column = passenger_position
        passenger = self.map[row][column]
        
        # keep track if somebody moved
        somebody_moved = False
        
        # if the passenger already moved this round, stop
        if self.moved[passenger] == 1:
            return somebody_moved
        
        # Case 1: they need to load their luggage
        if passenger.itinerary[0] == Itinerary.LOAD_LUGGAGE:
            # Case 1a: they did not finish the loading phase --> proceed with loading
            if passenger.remaining_loading_luggage:
                passenger.remaining_loading_luggage -= 1
                # the passenger moved (consumed their turn)
                self.moved[passenger] = 1
                somebody_moved = True
            # Case 1b: they did finish loading --> themselves, and possibly the people who had to move out to make space
            else:
                # they are not anymore in the "load_luggage" phase
                passenger.itinerary.pop(0)
                # they move
                somebody_moved = self.move(passenger_position)
                # their goal is to go_to_seat (if not already seated)
                if passenger.goal != Goal.SEATED:
                    passenger.goal = Goal.GO_TO_SEAT
                
                # if the two persons above didn't move this round, and were making space, they try to return to seat
                for r in range(row + 1, row + 3):
                    p = self.map[r][NUM_SEATS_PER_ROW_PER_SIDE]
                    if p and (not self.moved[p]) and p.goal == Goal.MAKE_SPACE:
                        p.goal = Goal.GO_TO_SEAT
                        somebody_moved = self.move((r, column))
        else:
            # the position the passenger needs to move to
            move_to = np.array(passenger_position) + np.array(passenger.itinerary[0])
            move_to = [int(i) for i in move_to]
            
            if self.map[move_to[0]][move_to[1]] == None:
                # move the passenger
                self.map[move_to[0]][move_to[1]] = passenger
                self.map[passenger_position[0]][passenger_position[1]] = None

                # remove that move from their itinerary
                passenger.itinerary.pop(0)

                # if they are in the right place, they're seated
                if passenger.row_no == move_to[0] and passenger.column_no == move_to[1]:
                    passenger.goal = Goal.SEATED
                    
                # the passenger moved
                self.moved[passenger] = 1
                somebody_moved = True
        
        return somebody_moved
        
    
    def move_if_space(self, passenger_position: tuple[int, int]) -> bool:
        """The passenger currently on the position passenger_position progresses or asks others to make space"""
                              
        row, column = passenger_position
        passenger = self.map[row][column]
        
        # will keep track if somebody moved
        somebody_moved = False
        
        # Case 1: they are one step before loading their luggage --> need to check more than one step in front for movement
        if len(passenger.itinerary) > 1 and passenger.itinerary[1] == Itinerary.LOAD_LUGGAGE:
            # need to have access to their place
            access = True
            for loc in passenger.needed_free:
                p = self.map[loc[0]][loc[1]] 
                # if there is somebody on a spot that is needed to be free
                if p:
                    # if they are not heading towards a place further away or to a place on the other row
                    not_further_away = abs(p.column_no - NUM_SEATS_PER_ROW_PER_SIDE) < abs(passenger.column_no - NUM_SEATS_PER_ROW_PER_SIDE)
                    not_on_the_other_side = ((p.column_no - NUM_SEATS_PER_ROW_PER_SIDE) * (passenger.column_no - NUM_SEATS_PER_ROW_PER_SIDE)) > 0
                    if not_further_away and not_on_the_other_side:
                        access = False
            
            # if they have access to their place, move
            if access:
                somebody_moved = self.move(passenger_position)
            # otherwise, ask the seated/go_to_place passengers to move
            else:
                # find the passengers that need to move
                seats_on_row = []
                for loc in passenger.needed_free:
                    p = self.map[loc[0]][loc[1]]
                    if p and p.goal != Goal.FIND_SEAT:
                        not_further_away = abs(p.column_no - NUM_SEATS_PER_ROW_PER_SIDE) < abs(passenger.column_no - NUM_SEATS_PER_ROW_PER_SIDE)
                        not_on_the_other_side = ((p.column_no - NUM_SEATS_PER_ROW_PER_SIDE) * (passenger.column_no - NUM_SEATS_PER_ROW_PER_SIDE)) > 0
                        if not_further_away and not_on_the_other_side:
                            seats_on_row += [loc]
                
                # the others need to make_space (if not already doing so)
                num_needed_free = len(seats_on_row)
                for i in range(num_needed_free):
                    loc = seats_on_row[i]
                    p = self.map[loc[0]][loc[1]]
                    # if they are not heading towards a further away place
                    if abs(p.column_no - NUM_SEATS_PER_ROW_PER_SIDE) < abs(passenger.column_no - NUM_SEATS_PER_ROW_PER_SIDE):
                        # 1) they need to make space
                        p.goal = Goal.MAKE_SPACE

                        # 2) their itinerary changes
                        # are there two persons that need to move out to make space and are the first ones?
                        # yes --> they have to move two places up the aisle
                        need_move_up = 1
                        if i == 0 and num_needed_free > 1 and self.map[seats_on_row[1][0]][seats_on_row[1][1]]:
                            need_move_up = 2

                        # the direction of movement is towards the aisle
                        out_direction = (NUM_SEATS_PER_ROW_PER_SIDE - p.column_no) / abs(p.column_no - NUM_SEATS_PER_ROW_PER_SIDE)
                        out_direction = int(out_direction)
                        # get to the aisle
                        p.itinerary = [(0, out_direction) for _i in range(abs(loc[1] - NUM_SEATS_PER_ROW_PER_SIDE))]
                        # go up - one space if they are the only one to move, two if somebody else needs to move out afterwards
                        p.itinerary += [(1, 0) for _i in range(need_move_up)]
                        # go back down
                        p.itinerary += [(-1, 0) for _i in range(need_move_up)]
                        # go to seat
                        p.itinerary += [(0, -out_direction) for _i in range(abs(NUM_SEATS_PER_ROW_PER_SIDE - p.column_no))]

                        # 3) they move, if they didn't already move in this round, they try to
                        if not self.moved[p]:
                            somebody_moved = self.move(loc)
        # Case 2: they only want to advance one step (either looking for their row, loading the luggage, or going to place)
        else:
            somebody_moved = self.move(passenger_position)
        
        return somebody_moved        
    
    def one_iteration(self) -> bool:
        """One time period passed.
        1) each agent that can get to their seat (either because just stored their luggage, or because they made space) does this
        2) each agent that needs to move away to make space for others does this
        3) each agent that wants to find their seat and put their luggage, and can get closer, gets closer
        4) if there is an empty seat at the entry of the aisle, a new agent enters the plane
        """
        
        # 0) So far, nobody moved in this iteration
        self.reset_moved()
        
        # Everybody can move at most once
        progress_this_iteration = False
        somebody_moved = True
        while somebody_moved:
            somebody_moved = False
            # 1) Move everybody that goes to their place --> order = front to back, sides to aisle
            for r in range(self.num_rows_map):
                for c_change in range(NUM_SEATS_PER_ROW_PER_SIDE, -1, -1):
                    cols = [NUM_SEATS_PER_ROW_PER_SIDE - c_change, NUM_SEATS_PER_ROW_PER_SIDE + c_change] if c_change else [NUM_SEATS_PER_ROW_PER_SIDE]
                    for c in cols:
                        if self.map[r][c] and self.map[r][c].goal == Goal.GO_TO_SEAT:
                            m = self.move_if_space((r,c))
                            if m:
                                somebody_moved = True

            # 2) Move those that are making space --> order = back to front, aisle to side
            for r in range(self.num_rows_map - 1, -1, -1):
                for c_change in range(0, NUM_SEATS_PER_ROW_PER_SIDE + 1):
                    for c in [NUM_SEATS_PER_ROW_PER_SIDE - c_change, NUM_SEATS_PER_ROW_PER_SIDE + c_change]:
                        if self.map[r][c] and self.map[r][c].goal == Goal.MAKE_SPACE:
                            m = self.move_if_space((r,c))
                            if m:
                                somebody_moved = True

            # 3) Move those that try to advance (find their seat) --> order = back to front
            for r in range(self.num_rows_map - 1, -1, -1):
                c = NUM_SEATS_PER_ROW_PER_SIDE
                if self.map[r][c] and self.map[r][c].goal == Goal.FIND_SEAT:
                    m = self.move_if_space((r,c))
                    if m:
                        somebody_moved = True

            # 4) Try to add a new person on the aisle
            added_passenger = self.new_passenger_in_plane()
            
            if somebody_moved or added_passenger:
                progress_this_iteration = True
        
        return progress_this_iteration

In [ ]:
from time import sleep

# how many seconds python should wait between printing the next step of the interation
sleep_time_between_prints: float = 1.0

from IPython.display import clear_output

class Simulation:
    def __init__ (self, random_seed: int, ordering_passengers = 'random', print_map = False) -> None:
        # 1)  whether there are intermediate prints of the map while simulating
        self.print_map: bool = print_map
        
        # 2) parameters for the simulation
        # the ordering of passengers - random by default
        self.ordering_passengers: str = ordering_passengers
        
        # the random seed
        self.seed: int = random_seed
        
        # 3) statistics kept during simulation
        # the number of iterations until the plane is full
        self.num_iterations: int = -1
        
    def order_passengers(self, plane: Plane) -> str | None:
        """Orders the passengers of a plane, given the ordering parameter."""
        
        if self.ordering_passengers == 'random':
            plane.boarding_order = deepcopy(plane.passengers)
            np.random.shuffle(plane.boarding_order)

        # ---- You should implement alternative orderings here -----
        elif self.ordering_passengers == 'BtF':
            # Your code goes here
            return 'Function not implemented'

        elif self.ordering_passengers == 'WMA':
            # Your code goes here
            return 'Function not implemented'

        elif self.ordering_passengers == 'precise':
            # Your code goes here
            return 'Function not implemented'

        elif self.ordering_passengers == 'your_ordering':
            # Your code goes here
            return 'Function not implemented'

        return 'Ordering not defined!'

    def run(self) -> int:
        """This function runs a simulation with the defined parameters"""
        
        # 0) set the random seed
        np.random.seed(self.seed)
        
        # 1) define a plane
        my_plane = Plane()
        
        # 2) order the passengers
        self.order_passengers(my_plane)
        
        # 3) run the simulation, updating the statistics
        still_arranging = True
        while still_arranging:
            self.num_iterations += 1
            if self.print_map:
                sleep(sleep_time_between_prints)
                clear_output(wait = True)
                print ("Iteration #", self.num_iterations)
                my_plane.print_map()
            still_arranging = my_plane.one_iteration()
        
        return self.num_iterations

The cell below serves as an example of how to run a simulation with a random seed of 79 that also prints the intermediate configurations of the plane. You will see the passengers (labeled by their seat numbers) and the plane positions. The spaces currently free in the plane are denoted by ‘___’ and the ones that are inaccessible to the agents are denoted by ‘xxx’.

If you print the plane configuration, make sure you use a small number of rows (otherwise you will wait for a long time).

---

**Important note for PyCharm users:**

If you are working on this notebook in PyCharm, you won't see any intermediate outputs and therefore cannot watch the simulations! To view the boarding/deboarding processes, open this notebook via Terminal/Browser and run the cell there.

---

In [ ]:
s = Simulation(random_seed=79, print_map=True)
s.run()

**Exercise 2.1 (5 points)**

In this exercise you need to identify and change the values of two parameters: the number of rows in the plane and the speed for the simulation visualisation. First, find the variable controlling the number of rows in the plane and change its value such that you have a larger plane (e.g. have 19 or 28 rows). Second, find the variable controlling how quickly the screenshots for the plane change and modify its value such that it is going twice as slow.

Don’t forget that some parameters depend on the value of others. Hence, when changing a base parameter, remember to also change the dependent one.

Please use the cell below to modify these variables accordingly. You can run the simulation again in the end to verify your solution. (If it takes too long to finish, you can also abort it.)

In [ ]:
### Use the space here to change the number of rows in the plane


**Exercise 2.2 (5 points)**

To avoid capturing isolated behavior, in ABMs using randomisation, we use multiple random seeds and report on averages. For example, if we want to see how many time units (i.e. iterations) it usually takes to get a complete boarding if people are entering the plane in a random order, you could take multiple random seeds, track the number of iterations taken for completing the boarding for each seed and report on the average.

Besides finding the mean time for a complete boarding, you can also find the standard deviation. Including the standard deviation, will allow you to see if the difference in efficiency between boarding orderings is significant or not. E.g. if the average boarding time for an ordering X is two standard deviation away from the random ordering, then this suggests we observe a significant (likely not obtained by chance) difference in performance between X and the random ordering.

Use the space below to define a function running multiple simulations and reporting on the mean and standard deviation. (**Hint:** the std and mean functions from numpy might help). Also, for convenience when grading, please use the 100 random seeds below.

In [ ]:
### Use the space here to write your function. Please use the random seeds generated below in "random_seeds"

np.random.seed(79)
random_seeds = [int(x * 100000) for x in np.random.random(100)]

def run_multiple(seeds=None, ordering='random'):
    # Your code goes here
    return 'Function not implemented'

In [ ]:
### If you want, you can use the space here to test your above-implemented function

run_multiple(random_seeds)

**Exercise 2.3 (20 points – 5 points per ordering)**

The idea of the exercise was to compare the efficiency of multiple orderings of passengers. In this exercise you will implement four orderings: 
- BtF (back-to-front). In this ordering, first the second half of the plane (e.g. rows 15–28) boards and next the front of the plane (rows 1–14).
- WMA (window – middle – aisle). In this ordering you should first form 3 groups: the group of passengers who should sit next to the window (window-passengers), those who stay in the middle (middle-passengers), and those who sit next to the aisle (aisle-passengers).  First the window passengers enter, next the middle ones, and last the aisle ones. Within the same group passengers are ordered at random.
-  Precise. In this ordering the agents are ordered in a very precise way. They enter the plane in 6 groups: window-left, middle-left, aisle-left, window-right, middle-right, aisle-right. Moreover, within each group, passengers are entering in reverse order of their row number.
- Last, please implement an ordering of your choice. Use the space below to describe this order.

To write the code for the orderings, please use the space allocated in the class `Simulation`, function `order_passengers`. The space below is only to describe the ordering for the last part.

**Describe your ordering here:**

...

**Exercise 2.4 (10 points)**

Use the space below to run simulations for each of the orderings. Report on the expected boarding time and standard deviation for each ordering. You can use your code from exercise 2.2 to do so.

In [ ]:
### Use the space here to run the simulations for the 4 orderings


**Exercise 2.5 (10 points)**

The amount of time passengers spend loading their hand-luggage makes a lot of difference in the amount of time until the boarding is complete, and therefore, also on the efficiency of different boarding methods. This exercise asks you to reflect on this. More precisely, you need to:
-	Find the variable that controls the time it takes one passenger to load their luggage. Change the value of this variable first to 0 (which corresponds to passengers having no luggage) and a larger number, such as 5. (2 points)
-	Use your function from exercise 2.2 to run the simulation with multiple random seeds for each ordering in exercise 2.3. Do so for both values of luggage loading time. (3 points)
-	What changes and what remains similar with the different loading times? Do the same methods remain efficient? Use the space below to draft your answer. (5 points)


In [ ]:
### Use this space for your code for the first two parts


**Your answer for the third part:**

...

**Exercise 2.6 (5 points)**

If you were the CEO of an airline, what ordering would you implement? Explain your choice.

## 3) Plane deboarding (35 points)

The passengers, of course, also have to deboard the plane. As before, below you will find the code for simulating the deboarding process. Please take some time to familiarise yourself with the code. Notice this is similar to the code for boarding (the one in exercise 2). 

In [ ]:
# parameter for the time it takes one passenger to unload their luggage
time_unloading_luggage = 1

In [ ]:
class DbPassenger:
    def __init__ (self, seat_no: int) -> None:
        # id = unique number between 1 and the number of seats
        # this corresponds to their seat number (-1 --> unassigned)
        self.id: int = seat_no
        
        # row number (starting from 1)
        self.row_no: int = int(seat_no / (NUM_SEATS_PER_ROW_PER_SIDE * NUM_SIDES)) + 1
        
        # the seat number within the row
        self.column_no: int = seat_no % (NUM_SEATS_PER_ROW_PER_SIDE * NUM_SIDES)
        # if the seat is across the aisle, add 1 to its number
        if self.column_no >= NUM_SEATS_PER_ROW_PER_SIDE:
            self.column_no += 1
                       
        # luggage un-loading status (initially, this is how long it takes an agent to unload their luggage)
        self.remaining_unloading_luggage: int = time_unloading_luggage
        
        # itinerary - what they have to do to get out of the plane
        self.itinerary: list[tuple[int, int] | Itinerary] = []

        # 1) get to the aisle
        if self.column_no < NUM_SEATS_PER_ROW_PER_SIDE:
            self.itinerary = ([(0, 1) for _i in range(NUM_SEATS_PER_ROW_PER_SIDE - self.column_no)])
        else:
            self.itinerary = ([(0, -1) for _i in range(self.column_no - NUM_SEATS_PER_ROW_PER_SIDE)])

        # 2) unload the luggage
        self.itinerary.extend([Itinerary.UNLOAD_LUGGAGE for _i in range(time_unloading_luggage)])

        # 3) advance on the aisle, i.e. move down, i.e. add (-1, 0) to their position
        self.itinerary.extend([(-1, 0) for _i in range(self.row_no + 1)])

    def __str__(self):
        return str(self.id)
    
    def __eq__ (self, other):
        """Two passengers are equal if they have the same seat"""
        if isinstance(other, DbPassenger):
            return self.id == other.id
        else:
            return False
    
    def __hash__(self):
        """Passengers are hashed by their id"""
        return self.id

In [ ]:
class DbPlane:
    def __init__(self, priority = 'front'):
        # passengers are labeled by their seat number
        self.passengers: list[DbPassenger] = [DbPassenger(i) for i in range(MUM_SEATS)]

        # frequency vector of passengers that moved this round (in one iteration a passenger can make at most one move)
        self.moved: dict[DbPassenger, int] = {self.passengers[i]:0 for i in range(MUM_SEATS)}
        
        # the type of priority at deboarding
        self.priority: str = priority
        
        # current plane occupation
        # None --> that space is empty
        # there is a row 0 (one extra space in front) and two rows at the end (for aisle space - no seats)
        self.num_rows_map = NUM_ROWS + 3
        self.num_cols_map = NUM_SEATS_PER_ROW_PER_SIDE * NUM_SIDES + 1

        # start with the empty configuration
        self.map: list[list[DbPassenger | None]] = [[None for _c in range(self.num_cols_map)] for _r in range(self.num_rows_map)]

        # fill in the seats
        for c in range(NUM_SEATS_PER_ROW_PER_SIDE):
            for r in range(NUM_ROWS - 1, -1, -1):
                self.map[r + 1][c] = self.passengers[r * 2 * NUM_SEATS_PER_ROW_PER_SIDE + c]
                self.map[r + 1][2 * NUM_SEATS_PER_ROW_PER_SIDE - c] = self.passengers[(r + 1) * 2 * NUM_SEATS_PER_ROW_PER_SIDE - c - 1]
    
    def print_map(self) -> None:
        """Prints the current plane map."""
        
        for r in range(self.num_rows_map):
            current_row = ""
            for c in range(self.num_cols_map):
                if self.map[r][c]:
                    l = len(str(self.map[r][c]))
                    spaces = "  "
                    for i in range(3 - l):
                        spaces += " "
                    current_row += spaces + str(self.map[r][c])
                elif r in [0, self.num_rows_map - 2, self.num_rows_map - 1] and c != NUM_SEATS_PER_ROW_PER_SIDE:
                    # these are unaccessible places
                    current_row += "  xxx"
                else:
                    current_row += "  ___"
            print(current_row)
    
    def reset_moved(self):
        """Resets moved to only 0"""
        
        for p in self.moved:
            self.moved[p] = 0
    
    def nobody_on_row(self, r: int) -> bool:
        """Checks if there are no passengers on row r.
        True --> nobody on row r
        False --> somebody is on row r"""
        
        found_somebody = False
        for c in range(self.num_cols_map):
            if self.map[r][c]:
                found_somebody = True
        
        return not found_somebody
    
    def move(self, passenger_position: tuple[int, int]) -> bool:
        """The passenger currently on the position passenger_position, and can start loading their luggage.
        They do one step in their itinerary."""
                              
        row, column = passenger_position
        passenger = self.map[row][column]
        
        # keep track if somebody moved
        somebody_moved = False
        
        # if the passenger already moved this round, stop
        if self.moved[passenger] == 1:
            return somebody_moved
        
        # Case 1: they need to unload their luggage
        if passenger.itinerary[0] == Itinerary.UNLOAD_LUGGAGE:
            if passenger.remaining_unloading_luggage:
                passenger.remaining_unloading_luggage -= 1
                passenger.itinerary.pop(0)
                self.moved[passenger] = 1
                somebody_moved = True
        # Case 2: they need to advance
        else:
            # the position the passenger needs to move to
            move_to = np.array(passenger_position) + np.array(passenger.itinerary[0])
            move_to = [int(i) for i in move_to]

            if self.map[move_to[0]][move_to[1]] == None:
                # move the passenger (unless they exit the plane)
                if move_to[0]:
                    self.map[move_to[0]][move_to[1]] = passenger
                self.map[passenger_position[0]][passenger_position[1]]  = None

                # remove that move from their itinerary
                passenger.itinerary.pop(0)
                    
                # the passenger moved
                self.moved[passenger] = 1
                somebody_moved = True
        
        return somebody_moved

    def advance_on_row(self, r: int) -> bool:
        progress = False
        for c_change in range(2, NUM_SEATS_PER_ROW_PER_SIDE + 1):
            cols = [NUM_SEATS_PER_ROW_PER_SIDE - c_change, NUM_SEATS_PER_ROW_PER_SIDE + c_change] if c_change else [NUM_SEATS_PER_ROW_PER_SIDE]
            for c in cols:
                if self.map[r][c]:
                    m = self.move((r,c))
                    if m:
                        progress = True
        return progress
    
    def one_iteration(self) -> bool | str:
        """One time period passed.
        1) each agent that can exit the plane exits the plane
        2) passengers that need to unload to this
        3) passengers advance on their row (without getting on the aisle)
        4) depending on the priority, agents advance
        """
        
        # 0) So far, nobody moved in this iteration
        self.reset_moved()
        
        # Everybody can move at most once
        progress_this_iteration = False
        somebody_moved = True
        
        while somebody_moved:
            somebody_moved = False

            # 1) passengers that need to unload do that (note, they must be on the aisle)
            for r in range(self.num_rows_map - 1, -1, -1):
                c = NUM_SEATS_PER_ROW_PER_SIDE
                if self.map[r][c] and self.map[r][c].itinerary[0] == Itinerary.UNLOAD_LUGGAGE:
                    m = self.move((r,c))
                    if m:
                        somebody_moved = True

            # 2) passengers can advance on their row (without getting on the aisle, do this)
            for r in range(self.num_rows_map):
                m = self.advance_on_row(r)
                if m:
                    somebody_moved = True

           # 3) the other agents move (depending on their priority)
            if self.priority == 'front':
                # Use the space here to write the rest of the moves for the agents with priority front
                # Hint: you might find the function nobody_on_row helpful
                return 'Function not implemented'

            elif self.priority == 'back':
                # Use the space here to write the rest of the moves for the agents with priority back
                return 'Function not implemented'

            if somebody_moved:
                progress_this_iteration = True
                 
        return progress_this_iteration

In [ ]:
# how many seconds python should wait between printing the next step of the interation
sleep_time_between_prints: float = 1.0

class DbSimulation:
    def __init__ (self, random_seed: int = 79, priority = 'front', print_map = False) -> None:
        # 1)  whether there are intermediate prints of the map while simulating
        self.print_map: bool = print_map
        
        # 2) parameters for the simulation
        # the ordering of passengers - random by default
        self.priority: str = priority
        
        # the random seed
        self.seed: int = random_seed
        
        # 3) statistics kept during simulation
        # the number of iterations until the plane is full
        self.num_iterations: int = -1
    
    def run(self) -> int:
        """This function runs a simulation with the defined parameters"""
        
        # 0) set the random seed
        np.random.seed(self.seed)
        
        # 1) define a plane
        pl = DbPlane(priority = self.priority)
        
        # 2) run the simulation, updating the statistics
        still_arranging = True
        while still_arranging:
            self.num_iterations += 1
            if self.print_map:
                sleep(sleep_time_between_prints)
                clear_output(wait = True)
                print ("Iteration #", self.num_iterations)
                pl.print_map()
            still_arranging = pl.one_iteration()
        
        return self.num_iterations

If you want, you can run the simulation using the following code snippet. Change the parameters or values according to your choice:
- Sleep time between prints
- Time (steps) to unload luggage
- Priority schema

---

**For PyCharm users:** You need to run this cell via Jupyter Notebook in the browser.

---

In [ ]:
# This is just for your convenience.
deboarding_simulation = DbSimulation(print_map = True, priority = 'front')
deboarding_simulation.run()

**Exercise 3.1 (20 points – 10 points per priority type)**

The idea of this part of the exercise is to compare the efficiency of multiple deboarding priorities. Please implement the following two types of priority:
-	Front. The people in the front of the plane have priority. Somebody will only advance down the aisle if the row in front (i.e. the one they are progressing to) has no passengers left.
-	Back. The people ready to go out (i.e. with their luggage already unloaded) have priority. A passenger will not get on the aisle if they block the way of a passenger already ready to go out. 

To write the code for the two priorities, please use the space allocated in the class `DbPlane`, function `one_iteration`.


**Exercise 3.2 (10 points)**

Now is time to run a simulation with the different priorities, report the results, and reflect on the differences. More precisely, please do the following: **For each of the two priority types and unloading-luggage-times** of 0, 1, and 5, **run the simulation** and **find the time** to completely deboard the plane.

You should hence do 6 runs:
- priority = front and unloading-luggage time = 0
- priority = front and unloading-luggage time = 1
- priority = front and unloading-luggage time = 5
- priority = back and unloading-luggage time = 0
- priority = back and unloading-luggage time = 1
- priority = back and unloading-luggage time = 5

You will receive 1 point for each.

Use the space below to reflect on the results. Don’t forget to say which priority you would encourage.

In [ ]:
### Use the space here to run the 6 simulations


**Reflect on the results:**

...

**Exercise 3.3 (5 points)**

One difference from the simulation onboarding is our lack of using random seeds. This time, we do not vary the seed. Why is this the case?